[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Drew-Thomson/boltzgen/blob/main/notebooks/colab_material_screen.ipynb)

# BoltzGen Cyclic Metal-Peptide Material Screen (Colab Edition)

Run cyclic metal-binding peptide assembly screens (modeled on `cyclic_AP_Zn_6mer.yaml`) using the modified BoltzGen pipeline on a **free Google Colab account**.

## Making the Most of Your Free Colab Quota
- **GPU for generation, CPU for analysis:** Select `Runtime -> Change runtime type -> T4 GPU` to run Cells **1–6**. Switch to `CPU` (`0 GPU quota`) for Cells **7–9** (cross-run dashboard, 3D viewer, re-filtering, ZIP export).
- **Start small, then scale:** Always verify backbone ring diameter and metal placement with **Backbones Only** (`~12–18 min`) before launching folding + counter-screens.
- **Keep `RESUME_EXISTING = True`:** Outputs write directly to Google Drive. If Colab disconnects mid-run, re-running Cell 4 picks up where it left off without losing completed structures.
- **Model weights stay on local SSD (`~4.5–5.5 GB`):** Downloading weights on Colab takes `~45–90s` per session and preserves your `15 GB` free Google Drive quota for actual design results (`~10–85 MB` per run).

| Preset | Designs (`backbones × seqs`) | Folding / Apo Samples | Max Counter-Screen Candidates | Est. T4 Time | Est. Drive Size |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Backbones Only** | `8 × 0` (diffusion only) | — | — | **~12–18 min** | ~1 MB |
| **Scout (Fast Sweep)** | `10 × 3` (30 seqs) | `2` / `2` | `10` | **~25–40 min** | ~12 MB |
| **Standard** | `20 × 4` (80 seqs) | `3` / `3` | `15` | **~70–100 min** | ~30 MB |
| **Full (`cyclic_AP_Zn_6mer`)** | `50 × 5` (250 seqs) | `5` / `5` | `30` | **~4.5–6 hr** *(resumes across sessions)* | ~85 MB |


In [ ]:
#@title 1. Check Hardware & Mount Google Drive
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_WORK_DIR = "/content/drive/MyDrive/boltzgen_runs"  #@param {type:"string"}
MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
CACHE_ON_DRIVE = False  #@param {type:"boolean"}

IN_COLAB = "google.colab" in sys.modules

# Check GPU via nvidia-smi without importing numpy/torch before Cell 2 install.
gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
HAS_GPU = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
if HAS_GPU:
    print(f"GPU detected: {gpu_probe.stdout.strip()}")
else:
    print("CPU-only runtime detected (0 GPU quota used -- Cells 7-9 work cleanly here).")

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    work_dir = Path(DRIVE_WORK_DIR)
else:
    work_dir = Path("./colab_boltzgen_runs").resolve()

work_dir.mkdir(parents=True, exist_ok=True)
print(f"Run output root: {work_dir}")


def _print_free_space(label: str, path: Path) -> None:
    if path.exists():
        usage = shutil.disk_usage(path)
        free_gb = usage.free / (1024 ** 3)
        total_gb = usage.total / (1024 ** 3)
        print(f"  {label}: {free_gb:.2f} GB free / {total_gb:.2f} GB total ({path})")


_print_free_space("Local SSD", Path("/content") if Path("/content").exists() else Path("."))
if work_dir.exists():
    _print_free_space("Output Storage", work_dir)

In [ ]:
#@title 2. Install Modified BoltzGen & Download Checkpoints (Run Before Importing NumPy/Pandas)
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Drew-Thomson/boltzgen.git"  #@param {type:"string"}
GIT_BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/boltzgen"  #@param {type:"string"}
FORCE_REINSTALL = False  #@param {type:"boolean"}
PRE_DOWNLOAD_WEIGHTS = True  #@param {type:"boolean"}
DESIGN_CHECKPOINT_MODE = "both"  #@param ["both", "diverse_only", "adherence_only"]

if "numpy" in sys.modules:
    print(
        "WARNING: numpy is already loaded in this Python session. If pip updates numpy, "
        "restart the runtime (Runtime -> Restart session) and re-run Cells 1-2."
    )

repo_path = Path(REPO_DIR) if Path("/content").exists() else Path(".").resolve()
if not repo_path.exists():
    print(f"Cloning {REPO_URL} (branch: {GIT_BRANCH}) -> {repo_path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(repo_path)],
        check=True,
    )
elif (repo_path / ".git").exists() and Path("/content").exists():
    print(f"Updating existing repo at {repo_path} ({GIT_BRANCH})...")
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)

needs_install = FORCE_REINSTALL or importlib.util.find_spec("boltzgen") is None
if not needs_install:
    spec = importlib.util.find_spec("boltzgen")
    origin = str(getattr(spec, "origin", ""))
    if str(repo_path) not in origin and Path("/content").exists():
        needs_install = True

if needs_install:
    print(f"Installing modified BoltzGen from {repo_path} (this takes ~1-2 min on first run)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_path)], check=True)
    print("BoltzGen installation complete.")
else:
    print("Modified BoltzGen already installed.")

if CACHE_ON_DRIVE:
    hf_cache = work_dir / "_hf_cache"
else:
    hf_cache = (
        Path("/content/.cache/huggingface")
        if Path("/content").exists()
        else Path.home() / ".cache/huggingface"
    )
hf_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_cache)
print(f"Hugging Face cache (HF_HOME): {hf_cache}")

CKPT_DIVERSE = "huggingface:boltzgen/boltzgen-1:boltzgen1_diverse.ckpt"
CKPT_ADHERENCE = "huggingface:boltzgen/boltzgen-1:boltzgen1_adherence.ckpt"


def get_selected_design_checkpoints(mode: str) -> list[str]:
    '''Map UI checkpoint mode to BoltzGen checkpoint artifact identifiers.'''
    if mode == "diverse_only":
        return [CKPT_DIVERSE]
    if mode == "adherence_only":
        return [CKPT_ADHERENCE]
    return [CKPT_DIVERSE, CKPT_ADHERENCE]


if PRE_DOWNLOAD_WEIGHTS:
    if HAS_GPU:
        artifacts = ["moldir", "inverse-fold", "folding"]
        if DESIGN_CHECKPOINT_MODE in {"both", "diverse_only"}:
            artifacts.append("design-diverse")
        if DESIGN_CHECKPOINT_MODE in {"both", "adherence_only"}:
            artifacts.append("design-adherence")
        print(f"Pre-downloading required artifacts (skipping affinity): {artifacts}")
        subprocess.run(["boltzgen", "download", *artifacts, "--cache", str(hf_cache)], check=True)
    else:
        print("CPU runtime: downloading lightweight moldir only (skipping GPU checkpoints).")
        subprocess.run(["boltzgen", "download", "moldir", "--cache", str(hf_cache)], check=True)

In [ ]:
#@title 3. Configure Cyclic Metal-Peptide Screen (Defaults Match cyclic_AP_Zn_6mer.yaml)
from pathlib import Path
import subprocess
import sys
from typing import Any
import yaml

# --- Run & Quota Preset ---
RUN_NAME = "Zn_AP_6mer_scout"  #@param {type:"string"}
PRESET = "Scout (10x3, fold=2, ~30m)"  #@param ["Backbones Only (8x0, ~15m)", "Scout (10x3, fold=2, ~30m)", "Standard (20x4, fold=3, ~80m)", "Full (50x5, fold=5, ~5h resume)", "Custom"]
RESUME_EXISTING = True  #@param {type:"boolean"}

# --- Cyclic Peptide Assembly ---
COPIES = 6  #@param {type:"integer"}
TARGET_RADIUS = 9.0  #@param {type:"number"}
SPACING_NOISE = 1.0  #@param {type:"number"}
ANTIPARALLEL_PROB = 1.0  #@param {type:"number"}
PEPTIDE_LENGTH = 20  #@param {type:"integer"}
SECONDARY_STRUCTURE = "H"  #@param ["H", "S", "L"]

# --- Metal / Ligand Placement & Counter-Screen ---
LIGAND_CCD = "ZN"  #@param {type:"string"}
LIGAND_PLACEMENT_MODE = "interface_anchored"  #@param ["interface_anchored", "z_axis", "none"]
LIGANDS_PER_PEPTIDE = 0.5  #@param {type:"number"}
ANCHOR_RESIDUE = "random"  #@param {type:"string"}
ANCHOR_DISTANCE = 3.5  #@param {type:"number"}
ANCHOR_DIRECTION = "interface"  #@param ["interface", "inward", "outward", "radial", "sidechain", "auto"]
AXIAL_COUNT = 2  #@param {type:"integer"}
AXIAL_Z_MIN = -6.0  #@param {type:"number"}
AXIAL_Z_MAX = 6.0  #@param {type:"number"}
MIN_LIGAND_DISTANCE = 3.5  #@param {type:"number"}
LIGAND_COUNTER_SCREEN = True  #@param {type:"boolean"}

# --- Custom Counts & Inverse Folding ---
CUSTOM_NUM_DESIGNS = 10  #@param {type:"integer"}
CUSTOM_SEQS_PER_BACKBONE = 3  #@param {type:"integer"}
CUSTOM_FOLD_SAMPLES = 2  #@param {type:"integer"}
CUSTOM_MAX_TARGET_CANDIDATES = 10  #@param {type:"integer"}
INVERSE_TEMP = 0.15  #@param {type:"number"}
BACKBONE_NOISE = 0.1  #@param {type:"number"}
REPETITION_PENALTY = 0.5  #@param {type:"number"}
AVOID_AA = "M"  #@param {type:"string"}
AA_BIAS = ""  #@param {type:"string"}
BUDGET = 20  #@param {type:"integer"}

# --- Optional Raw YAML Override (leave empty to use the form above) ---
RAW_YAML_OVERRIDE = ""  #@param {type:"string"}

PRESET_TABLE: dict[str, dict[str, int | bool]] = {
    "Backbones Only (8x0, ~15m)": {
        "num_designs": 8,
        "seqs_per_backbone": 1,
        "fold_samples": 1,
        "max_target_candidates": 8,
        "design_only": True,
    },
    "Scout (10x3, fold=2, ~30m)": {
        "num_designs": 10,
        "seqs_per_backbone": 3,
        "fold_samples": 2,
        "max_target_candidates": 10,
        "design_only": False,
    },
    "Standard (20x4, fold=3, ~80m)": {
        "num_designs": 20,
        "seqs_per_backbone": 4,
        "fold_samples": 3,
        "max_target_candidates": 15,
        "design_only": False,
    },
    "Full (50x5, fold=5, ~5h resume)": {
        "num_designs": 50,
        "seqs_per_backbone": 5,
        "fold_samples": 5,
        "max_target_candidates": 30,
        "design_only": False,
    },
}


def resolve_preset_settings(
    preset_name: str,
    custom_num_designs: int,
    custom_seqs: int,
    custom_fold_samples: int,
    custom_max_cands: int,
) -> dict[str, int | bool]:
    '''Return sampling and folding counts for the selected quota preset.'''
    if preset_name in PRESET_TABLE:
        return dict(PRESET_TABLE[preset_name])
    return {
        "num_designs": int(custom_num_designs),
        "seqs_per_backbone": int(custom_seqs),
        "fold_samples": int(custom_fold_samples),
        "max_target_candidates": int(custom_max_cands),
        "design_only": False,
    }


def build_cyclic_material_config(
    *,
    copies: int,
    target_radius: float,
    spacing_noise: float,
    antiparallel_prob: float,
    peptide_length: int,
    secondary_structure: str,
    ligand_ccd: str,
    ligand_placement_mode: str,
    ligands_per_peptide: float,
    anchor_residue: str,
    anchor_distance: float,
    anchor_direction: str,
    axial_count: int,
    axial_z_min: float,
    axial_z_max: float,
    min_ligand_distance: float,
    ligand_counter_screen: bool,
    num_designs: int,
    seqs_per_backbone: int,
    max_target_candidates: int,
    inverse_temp: float,
    backbone_noise: float,
    repetition_penalty: float,
    avoid_aa: str,
    aa_bias: str,
) -> dict[str, Any]:
    '''Build a material_builder YAML config dict for a cyclic metal-peptide screen.'''
    asym_unit: list[dict[str, Any]] = [
        {
            "type": "protein",
            "name": "helical_peptide",
            "length": int(peptide_length),
            "secondary_structure": str(secondary_structure),
        }
    ]
    use_ligand = ligand_placement_mode != "none" and bool(ligand_ccd.strip())
    if ligand_placement_mode == "interface_anchored" and use_ligand:
        parsed_anchor: int | str = (
            int(anchor_residue)
            if str(anchor_residue).strip().isdigit()
            else str(anchor_residue).strip()
        )
        asym_unit.append(
            {
                "type": "ligand",
                "ccd": str(ligand_ccd).strip().upper(),
                "ligands_per_peptide": float(ligands_per_peptide),
                "attach_to": "helical_peptide",
                "anchor_residue": parsed_anchor,
                "anchor_distance": float(anchor_distance),
                "anchor_direction": str(anchor_direction),
            }
        )
    elif ligand_placement_mode == "z_axis" and use_ligand:
        asym_unit.append(
            {
                "type": "ligand",
                "ccd": str(ligand_ccd).strip().upper(),
                "placement": "z_axis",
                "constraint": "z_axis",
                "count": int(axial_count),
                "distribution": "random_even",
                "z_range": [float(axial_z_min), float(axial_z_max)],
                "min_distance": float(min_ligand_distance),
            }
        )

    cfg: dict[str, Any] = {
        "copies": int(copies),
        "topology": "cyclic",
        "target_radius": float(target_radius),
        "antiparallel_prob": float(antiparallel_prob),
        "num_designs": int(num_designs),
        "seqs_per_backbone": int(seqs_per_backbone),
        "spacing_noise": float(spacing_noise),
        "avoid_aa": str(avoid_aa),
        "inverse_temp": float(inverse_temp),
        "backbone_noise": float(backbone_noise),
        "repetition_penalty": float(repetition_penalty),
        "ligand_counter_screen": bool(ligand_counter_screen and use_ligand),
        "max_target_candidates": int(max_target_candidates),
        "min_ligand_distance": float(min_ligand_distance),
        "asym_unit": asym_unit,
    }
    if aa_bias.strip():
        cfg["aa_bias"] = aa_bias.strip()
    return cfg


def estimate_t4_minutes(
    *,
    num_designs: int,
    seqs_per_backbone: int,
    fold_samples: int,
    max_target_candidates: int,
    ligand_counter_screen: bool,
    design_only: bool,
    num_design_checkpoints: int = 2,
) -> float:
    '''Approximate Colab Tesla T4 runtime in minutes (calibrated from RTX 4080 Zn_AP_6mer250).'''
    ckpt_overhead = 1.5 * max(1, num_design_checkpoints)
    diffusion_min = ckpt_overhead + num_designs * 1.75
    if design_only:
        return diffusion_min
    total_seqs = num_designs * seqs_per_backbone
    ifold_min = 1.0 + total_seqs * 0.05
    fold_min = 1.5 + total_seqs * fold_samples * 0.14
    cs_min = 0.0
    if ligand_counter_screen:
        est_cands = min(total_seqs, max_target_candidates)
        cs_min = 1.5 + est_cands * fold_samples * 0.13
    analysis_filter_min = 1.5 + total_seqs * 0.02
    return diffusion_min + ifold_min + fold_min + cs_min + analysis_filter_min


def prepare_run_directory(
    run_root: Path,
    run_name: str,
    cfg: dict[str, Any],
    *,
    repo_root: Path,
) -> tuple[Path, Path]:
    '''Write input_material_config.yaml and dry-validate material_spec.yaml generation.'''
    run_dir = run_root / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = run_dir / "input_material_config.yaml"
    with open(cfg_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    builder_script = repo_root / "material_builder.py"
    subprocess.run(
        [sys.executable, str(builder_script), "--config", str(cfg_path), "--output_dir", str(run_dir)],
        check=True,
    )
    return run_dir, cfg_path


preset_cfg = resolve_preset_settings(
    PRESET,
    CUSTOM_NUM_DESIGNS,
    CUSTOM_SEQS_PER_BACKBONE,
    CUSTOM_FOLD_SAMPLES,
    CUSTOM_MAX_TARGET_CANDIDATES,
)

if RAW_YAML_OVERRIDE.strip():
    active_config = yaml.safe_load(RAW_YAML_OVERRIDE)
    if not isinstance(active_config, dict):
        raise ValueError("RAW_YAML_OVERRIDE must parse to a YAML mapping.")
    print("Using RAW_YAML_OVERRIDE instead of form fields.")
else:
    active_config = build_cyclic_material_config(
        copies=COPIES,
        target_radius=TARGET_RADIUS,
        spacing_noise=SPACING_NOISE,
        antiparallel_prob=ANTIPARALLEL_PROB,
        peptide_length=PEPTIDE_LENGTH,
        secondary_structure=SECONDARY_STRUCTURE,
        ligand_ccd=LIGAND_CCD,
        ligand_placement_mode=LIGAND_PLACEMENT_MODE,
        ligands_per_peptide=LIGANDS_PER_PEPTIDE,
        anchor_residue=ANCHOR_RESIDUE,
        anchor_distance=ANCHOR_DISTANCE,
        anchor_direction=ANCHOR_DIRECTION,
        axial_count=AXIAL_COUNT,
        axial_z_min=AXIAL_Z_MIN,
        axial_z_max=AXIAL_Z_MAX,
        min_ligand_distance=MIN_LIGAND_DISTANCE,
        ligand_counter_screen=LIGAND_COUNTER_SCREEN,
        num_designs=int(preset_cfg["num_designs"]),
        seqs_per_backbone=int(preset_cfg["seqs_per_backbone"]),
        max_target_candidates=int(preset_cfg["max_target_candidates"]),
        inverse_temp=INVERSE_TEMP,
        backbone_noise=BACKBONE_NOISE,
        repetition_penalty=REPETITION_PENALTY,
        avoid_aa=AVOID_AA,
        aa_bias=AA_BIAS,
    )

selected_ckpts = get_selected_design_checkpoints(DESIGN_CHECKPOINT_MODE)
est_min = estimate_t4_minutes(
    num_designs=int(active_config.get("num_designs", 10)),
    seqs_per_backbone=int(active_config.get("seqs_per_backbone", 1)),
    fold_samples=int(preset_cfg["fold_samples"]),
    max_target_candidates=int(active_config.get("max_target_candidates", 10)),
    ligand_counter_screen=bool(active_config.get("ligand_counter_screen", False)),
    design_only=bool(preset_cfg["design_only"]),
    num_design_checkpoints=len(selected_ckpts),
)

active_run_dir, active_cfg_path = prepare_run_directory(
    work_dir, RUN_NAME, active_config, repo_root=repo_path
)
print("\n--- Resolved Material Config ---")
print(yaml.safe_dump(active_config, sort_keys=False))
print(f"Config saved to: {active_cfg_path}")
print(f"Estimated Colab T4 wall-clock time: ~{est_min:.0f} min ({est_min / 60.0:.2f} hr)")

In [ ]:
#@title 4. Run Pipeline (Live Logs, --reuse Resume, Optional Auto-Disconnect)
import subprocess
import sys
import time
from pathlib import Path

NUM_WORKERS = 2  #@param {type:"integer"}
USE_KERNELS = "auto"  #@param ["auto", "true", "false"]
AUTO_DISCONNECT_AFTER_RUN = False  #@param {type:"boolean"}


def build_material_runner_cmd(
    *,
    repo_root: Path,
    cfg_path: Path,
    run_dir: Path,
    fold_samples: int,
    ligand_counter_screen: bool,
    design_only: bool,
    resume_existing: bool,
    budget: int,
    num_workers: int,
    use_kernels: str,
    design_checkpoints: list[str],
    cache_dir: Path,
    steps_override: list[str] | None = None,
) -> list[str]:
    '''Assemble the material_builder.py CLI command for Colab execution.'''
    cmd = [
        sys.executable,
        str(repo_root / "material_builder.py"),
        "--config",
        str(cfg_path),
        "--output_dir",
        str(run_dir),
        "--run",
        "--num_workers",
        str(num_workers),
        "--use_kernels",
        str(use_kernels),
        "--budget",
        str(budget),
        "--cache",
        str(cache_dir),
    ]
    if resume_existing:
        cmd.append("--reuse")
    if design_checkpoints:
        cmd.extend(["--design_checkpoints", *design_checkpoints])
    if steps_override:
        cmd.extend(["--steps", *steps_override])
    elif design_only:
        cmd.extend(["--steps", "design"])
    else:
        cmd.extend(["--config", "folding", f"diffusion_samples={fold_samples}"])
        if ligand_counter_screen:
            cmd.extend(["--config", "counter_screen_folding", f"diffusion_samples={fold_samples}"])
    return cmd


def stream_command(cmd: list[str]) -> int:
    '''Run a command while streaming combined stdout/stderr live.'''
    print("Executing:", " ".join(cmd))
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    elapsed = (time.time() - t0) / 60.0
    print(f"\nFinished with exit code {rc} in {elapsed:.1f} min.")
    return rc


def maybe_unassign_colab_runtime(enabled: bool) -> None:
    '''Disconnect Colab GPU runtime after completion to preserve free quota.'''
    if enabled and IN_COLAB:
        print("Auto-disconnect enabled: releasing Colab runtime in 15s (Interrupt to cancel)...")
        time.sleep(15)
        from google.colab import runtime
        runtime.unassign()


if not HAS_GPU:
    raise RuntimeError("No GPU detected. Switch to T4 GPU (Runtime -> Change runtime type) before running Cell 4.")

run_cmd = build_material_runner_cmd(
    repo_root=repo_path,
    cfg_path=active_cfg_path,
    run_dir=active_run_dir,
    fold_samples=int(preset_cfg["fold_samples"]),
    ligand_counter_screen=bool(active_config.get("ligand_counter_screen", False)),
    design_only=bool(preset_cfg["design_only"]),
    resume_existing=RESUME_EXISTING,
    budget=BUDGET,
    num_workers=NUM_WORKERS,
    use_kernels=USE_KERNELS,
    design_checkpoints=selected_ckpts,
    cache_dir=hf_cache,
)

try:
    exit_code = stream_command(run_cmd)
    if exit_code != 0:
        raise RuntimeError(f"BoltzGen exited with code {exit_code}. Re-run with RESUME_EXISTING=True to continue.")
finally:
    maybe_unassign_colab_runtime(AUTO_DISCONNECT_AFTER_RUN)

In [ ]:
#@title 5. Small-Run Parameter Sweep Helper (Edit Grid Below)
import copy
import itertools
from typing import Any

SWEEP_PREFIX = "Zn_AP_6mer_sweep"  #@param {type:"string"}
RUN_SWEEP = False  #@param {type:"boolean"}
SKIP_COMPLETED = True  #@param {type:"boolean"}
AUTO_DISCONNECT_AFTER_SWEEP = False  #@param {type:"boolean"}

# Edit this dictionary to sweep 1D or small 2D grids around Cell 3's base config.
# Supported keys: top-level YAML keys (e.g. target_radius, inverse_temp, spacing_noise,
# backbone_noise, avoid_aa) or ligand keys: ligand_ccd, anchor_distance, anchor_direction.
SWEEP_PARAM_GRID: dict[str, list[Any]] = {
    "target_radius": [8.0, 9.0, 10.0],
    # "anchor_distance": [3.0, 3.5],
}


def apply_sweep_point(base_cfg: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:
    '''Return a deep-copied config dict with sweep overrides applied cleanly.'''
    cfg = copy.deepcopy(base_cfg)
    for k, v in overrides.items():
        if k in {"ligand_ccd", "anchor_distance", "anchor_direction", "anchor_residue", "ligands_per_peptide"}:
            target_k = "ccd" if k == "ligand_ccd" else k
            for item in cfg.get("asym_unit", []):
                if str(item.get("type", "")).lower() == "ligand":
                    item[target_k] = v
        else:
            cfg[k] = v
    return cfg


keys = list(SWEEP_PARAM_GRID.keys())
combos = [dict(zip(keys, vals)) for vals in itertools.product(*(SWEEP_PARAM_GRID[k] for k in keys))]
per_run_min = estimate_t4_minutes(
    num_designs=int(active_config.get("num_designs", 10)),
    seqs_per_backbone=int(active_config.get("seqs_per_backbone", 1)),
    fold_samples=int(preset_cfg["fold_samples"]),
    max_target_candidates=int(active_config.get("max_target_candidates", 10)),
    ligand_counter_screen=bool(active_config.get("ligand_counter_screen", False)),
    design_only=bool(preset_cfg["design_only"]),
    num_design_checkpoints=len(selected_ckpts),
)

print(f"Planned sweep: {len(combos)} run(s), ~{per_run_min:.0f} min each (~{len(combos) * per_run_min:.0f} min total on T4):")
sweep_jobs: list[tuple[str, dict[str, Any]]] = []
for combo in combos:
    tag = "_".join(f"{k}-{v}".replace(".", "p") for k, v in combo.items())
    rname = f"{SWEEP_PREFIX}_{tag}"
    rcfg = apply_sweep_point(active_config, combo)
    done_marker = work_dir / rname / "final_ranked_designs" / "all_designs_metrics.csv"
    status = "DONE (will skip)" if (SKIP_COMPLETED and done_marker.exists()) else "PENDING"
    print(f"  - {rname}: {combo} [{status}]")
    sweep_jobs.append((rname, rcfg))

if RUN_SWEEP:
    if not HAS_GPU:
        raise RuntimeError("Switch to T4 GPU runtime before setting RUN_SWEEP=True.")
    try:
        for rname, rcfg in sweep_jobs:
            done_marker = work_dir / rname / "final_ranked_designs" / "all_designs_metrics.csv"
            if SKIP_COMPLETED and done_marker.exists():
                print(f"Skipping completed run: {rname}")
                continue
            print(f"\n=== Running sweep point: {rname} ===")
            rdir, rcfg_path = prepare_run_directory(work_dir, rname, rcfg, repo_root=repo_path)
            cmd = build_material_runner_cmd(
                repo_root=repo_path,
                cfg_path=rcfg_path,
                run_dir=rdir,
                fold_samples=int(preset_cfg["fold_samples"]),
                ligand_counter_screen=bool(rcfg.get("ligand_counter_screen", False)),
                design_only=bool(preset_cfg["design_only"]),
                resume_existing=True,
                budget=BUDGET,
                num_workers=NUM_WORKERS,
                use_kernels=USE_KERNELS,
                design_checkpoints=selected_ckpts,
                cache_dir=hf_cache,
            )
            rc = stream_command(cmd)
            if rc != 0:
                raise RuntimeError(f"Sweep run {rname} failed with code {rc}.")
    finally:
        maybe_unassign_colab_runtime(AUTO_DISCONNECT_AFTER_SWEEP)
else:
    print("\nDry preview only. Set RUN_SWEEP=True to execute.")

In [ ]:
#@title 6. Quota-Saver: Reuse Backbones for Sequence-Only Sweeps (0 Diffusion Cost)
import copy
import shutil
import yaml

SOURCE_RUN_NAME = "Zn_AP_6mer_scout"  #@param {type:"string"}
TARGET_RUN_NAME = "Zn_AP_6mer_seq_temp02"  #@param {type:"string"}
NEW_SEQS_PER_BACKBONE = 5  #@param {type:"integer"}
NEW_INVERSE_TEMP = 0.2  #@param {type:"number"}
NEW_BACKBONE_NOISE = 0.1  #@param {type:"number"}
NEW_REPETITION_PENALTY = 0.5  #@param {type:"number"}
NEW_AVOID_AA = "M"  #@param {type:"string"}
NEW_AA_BIAS = ""  #@param {type:"string"}
SEQ_FOLD_SAMPLES = 2  #@param {type:"integer"}
SEQ_MAX_CANDIDATES = 10  #@param {type:"integer"}
RUN_SEQ_SWEEP = False  #@param {type:"boolean"}
AUTO_DISCONNECT_AFTER_SEQ_SWEEP = False  #@param {type:"boolean"}

src_dir = work_dir / SOURCE_RUN_NAME
dst_dir = work_dir / TARGET_RUN_NAME
src_designs = src_dir / "intermediate_designs"
src_cfg_file = src_dir / "input_material_config.yaml"

if RUN_SEQ_SWEEP:
    if not HAS_GPU:
        raise RuntimeError("Switch to T4 GPU runtime before running sequence-only sweeps.")
    if not src_designs.exists() or not any(src_designs.glob("*.cif")):
        raise FileNotFoundError(f"No backbones found in {src_designs}")

    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_designs = dst_dir / "intermediate_designs"
    if not dst_designs.exists():
        shutil.copytree(src_designs, dst_designs)
    for fname in ("material_spec.yaml", "material_layout.json"):
        if (src_dir / fname).exists() and not (dst_dir / fname).exists():
            shutil.copy2(src_dir / fname, dst_dir / fname)

    if src_cfg_file.exists():
        with open(src_cfg_file, "r") as f:
            seq_cfg = yaml.safe_load(f)
    else:
        seq_cfg = copy.deepcopy(active_config)

    seq_cfg["seqs_per_backbone"] = int(NEW_SEQS_PER_BACKBONE)
    seq_cfg["inverse_temp"] = float(NEW_INVERSE_TEMP)
    seq_cfg["backbone_noise"] = float(NEW_BACKBONE_NOISE)
    seq_cfg["repetition_penalty"] = float(NEW_REPETITION_PENALTY)
    seq_cfg["avoid_aa"] = str(NEW_AVOID_AA)
    seq_cfg["max_target_candidates"] = int(SEQ_MAX_CANDIDATES)
    if NEW_AA_BIAS.strip():
        seq_cfg["aa_bias"] = NEW_AA_BIAS.strip()
    else:
        seq_cfg.pop("aa_bias", None)

    dst_cfg_path = dst_dir / "input_material_config.yaml"
    with open(dst_cfg_path, "w") as f:
        yaml.safe_dump(seq_cfg, f, sort_keys=False)

    use_lcs = bool(seq_cfg.get("ligand_counter_screen", False))
    seq_steps = ["inverse_folding", "folding", "analysis"]
    if use_lcs:
        seq_steps.extend(
            [
                "filter_target_candidates",
                "prepare_counter_screens",
                "counter_screen_folding",
                "merge_counter_screen_metrics",
            ]
        )
    seq_steps.append("filtering")

    cmd = build_material_runner_cmd(
        repo_root=repo_path,
        cfg_path=dst_cfg_path,
        run_dir=dst_dir,
        fold_samples=SEQ_FOLD_SAMPLES,
        ligand_counter_screen=use_lcs,
        design_only=False,
        resume_existing=True,
        budget=BUDGET,
        num_workers=NUM_WORKERS,
        use_kernels=USE_KERNELS,
        design_checkpoints=selected_ckpts,
        cache_dir=hf_cache,
        steps_override=seq_steps,
    )
    try:
        rc = stream_command(cmd)
        if rc != 0:
            raise RuntimeError(f"Sequence-only sweep failed with code {rc}.")
    finally:
        maybe_unassign_colab_runtime(AUTO_DISCONNECT_AFTER_SEQ_SWEEP)
else:
    print("Set RUN_SEQ_SWEEP=True to inverse-fold & refold existing backbones without diffusion.")

In [ ]:
#@title 7. Cross-Run Dashboard & Comparison Plots (CPU-Only Compatible)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

np.random.seed(42)

summary_rows: list[dict] = []
all_dfs: list[pd.DataFrame] = []

for run_dir in sorted(work_dir.iterdir()):
    if not run_dir.is_dir() or run_dir.name.startswith("_"):
        continue
    metrics_path = run_dir / "final_ranked_designs" / "all_designs_metrics.csv"
    if not metrics_path.exists():
        continue

    df = pd.read_csv(metrics_path)
    df["run_name"] = run_dir.name
    all_dfs.append(df)

    cfg = {}
    cfg_path = run_dir / "input_material_config.yaml"
    if cfg_path.exists():
        with open(cfg_path, "r") as f:
            cfg = yaml.safe_load(f) or {}

    sort_col = "final_rank" if "final_rank" in df.columns else ("max_rank" if "max_rank" in df.columns else None)
    df_sorted = df.sort_values(sort_col) if sort_col else df
    top = df_sorted.iloc[0] if not df_sorted.empty else {}

    if "num_filters_passed" in df.columns and not df.empty:
        passed = int((df["num_filters_passed"] == df["num_filters_passed"].max()).sum())
    else:
        passed = len(df)
    summary_rows.append(
        {
            "run_name": run_dir.name,
            "n_designs": len(df),
            "n_max_passed": passed,
            "target_radius": cfg.get("target_radius"),
            "inverse_temp": cfg.get("inverse_temp"),
            "best_holo_apo_rmsd": round(float(top.get("holo_apo_rmsd", np.nan)), 3) if "holo_apo_rmsd" in df.columns else np.nan,
            "best_complex_plddt": round(float(top.get("complex_plddt", np.nan)), 3) if "complex_plddt" in df.columns else np.nan,
            "best_ligand_iptm": round(float(top.get("ligand_iptm", np.nan)), 3) if "ligand_iptm" in df.columns else np.nan,
            "best_coord_num": round(float(top.get("ligand_min_coord_number", np.nan)), 2) if "ligand_min_coord_number" in df.columns else np.nan,
            "best_sym_rmsd": round(float(top.get("sym_chain_rmsd", np.nan)), 3) if "sym_chain_rmsd" in df.columns else np.nan,
            "top_sequence": top.get("designed_chain_sequence", ""),
        }
    )

if not summary_rows:
    print(f"No completed runs with final_ranked_designs/all_designs_metrics.csv found in {work_dir}.")
else:
    summary_df = pd.DataFrame(summary_rows)
    try:
        from IPython.display import display
        display(summary_df)
    except ImportError:
        print(summary_df.to_string(index=False))

    combined_df = pd.concat(all_dfs, ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)

    # Panel A: Holo-vs-Apo RMSD vs Ligand iPTM (colored by complex_plddt)
    ax = axes[0]
    x_col = "ligand_iptm" if "ligand_iptm" in combined_df.columns else "complex_plddt"
    y_col = "holo_apo_rmsd" if "holo_apo_rmsd" in combined_df.columns else "local_rmsd"
    plot_df = (
        combined_df.dropna(subset=[x_col, y_col])
        if x_col in combined_df.columns and y_col in combined_df.columns
        else pd.DataFrame()
    )
    if not plot_df.empty:
        color_vals = plot_df["complex_plddt"] if "complex_plddt" in plot_df.columns else np.zeros(len(plot_df))
        sc = ax.scatter(
            plot_df[x_col],
            plot_df[y_col],
            c=color_vals,
            cmap="viridis",
            alpha=0.8,
            edgecolors="black",
            linewidths=0.4,
            label="Designs",
        )
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("complex_plddt")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title("Ligand Binding vs. Holo-Apo Conformational Shift")
        ax.legend(loc="best")
        ax.grid(alpha=0.25)
    else:
        ax.text(0.5, 0.5, "Counter-screen / ligand metrics unavailable", ha="center", va="center")
        ax.set_title("Ligand Metrics")

    # Panel B: Per-run top complex_plddt and ligand_iptm comparison
    ax2 = axes[1]
    x_idx = np.arange(len(summary_df))
    width = 0.35
    colors = plt.get_cmap("viridis")([0.25, 0.75])
    plddt_vals = summary_df["best_complex_plddt"].fillna(0.0).to_numpy(dtype=float)
    iptm_vals = summary_df["best_ligand_iptm"].fillna(0.0).to_numpy(dtype=float)
    ax2.bar(x_idx - width / 2, plddt_vals, width=width, color=colors[0], label="Top complex_plddt")
    ax2.bar(x_idx + width / 2, iptm_vals, width=width, color=colors[1], label="Top ligand_iptm")
    ax2.set_xticks(x_idx)
    ax2.set_xticklabels(summary_df["run_name"], rotation=25, ha="right")
    ax2.set_ylabel("Score (0 - 1)")
    ax2.set_ylim(0.0, 1.05)
    ax2.set_title("Top Candidate Confidence Across Runs")
    ax2.legend(loc="best")
    ax2.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.show()

In [ ]:
#@title 8. Interactive 3D Holo vs. Apo Structure Viewer (CPU-Only Compatible)
import importlib.util
import subprocess
import sys
import pandas as pd

VIEW_RUN_NAME = "Zn_AP_6mer_scout"  #@param {type:"string"}
DESIGN_RANK_OR_ID = "1"  #@param {type:"string"}
SHOW_APO_SIDE_BY_SIDE = True  #@param {type:"boolean"}
COORD_RADIUS = 3.2  #@param {type:"number"}

if importlib.util.find_spec("py3Dmol") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py3Dmol"], check=True)
import py3Dmol

view_dir = work_dir / VIEW_RUN_NAME
metrics_csv = view_dir / "final_ranked_designs" / "all_designs_metrics.csv"

design_id = DESIGN_RANK_OR_ID.strip()
row_info = None
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    sort_col = "final_rank" if "final_rank" in df.columns else ("max_rank" if "max_rank" in df.columns else None)
    if sort_col:
        df = df.sort_values(sort_col).reset_index(drop=True)
    if design_id.isdigit():
        idx = int(design_id) - 1
        if 0 <= idx < len(df):
            row_info = df.iloc[idx]
            design_id = str(row_info["id"])
    elif "id" in df.columns:
        matched = df[df["id"].astype(str) == design_id]
        if not matched.empty:
            row_info = matched.iloc[0]

holo_candidates = [
    view_dir / "intermediate_designs_inverse_folded" / "refold_cif" / f"{design_id}.cif",
    view_dir / "intermediate_designs" / f"{design_id}.cif",
]
holo_cif = next((p for p in holo_candidates if p.exists()), None)
if holo_cif is None and (view_dir / "intermediate_designs").exists():
    cifs = sorted((view_dir / "intermediate_designs").glob("*.cif"))
    if cifs:
        holo_cif = cifs[0]
        design_id = holo_cif.stem

if holo_cif is None:
    raise FileNotFoundError(f"Could not find a CIF structure for '{DESIGN_RANK_OR_ID}' in {view_dir}.")

apo_candidates = [
    view_dir / "ligand_counter_screens" / "refold_cif" / f"{design_id}_apo.cif",
    view_dir / "ligand_counter_screens" / f"{design_id}_apo.cif",
]
apo_cif = next((p for p in apo_candidates if p.exists()), None)

if row_info is not None:
    cols = [
        "id",
        "final_rank",
        "holo_apo_rmsd",
        "complex_plddt",
        "ligand_iptm",
        "ligand_min_coord_number",
        "ligand_min_coord_n_chains",
        "sym_chain_rmsd",
        "designed_chain_sequence",
    ]
    print("Selected design metrics:")
    for c in cols:
        if c in row_info.index:
            print(f"  {c}: {row_info[c]}")
else:
    print(f"Viewing structure: {holo_cif}")


def _style_metal_assembly(viewer: "py3Dmol.view", panel: tuple[int, int] | None = None) -> None:
    kw = {"viewer": panel} if panel is not None else {}
    viewer.setStyle({"cartoon": {"colorscheme": "chain"}}, **kw)
    viewer.addStyle(
        {"hetflag": True},
        {"sphere": {"radius": 1.15, "color": "#f59e0b"}},
        **kw,
    )
    viewer.addStyle(
        {"within": {"distance": float(COORD_RADIUS), "sel": {"hetflag": True}}},
        {"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}},
        **kw,
    )
    viewer.zoomTo(**kw)


if SHOW_APO_SIDE_BY_SIDE and apo_cif is not None:
    print(f"Left: Holo refold ({holo_cif.name}) | Right: Apo counter-screen ({apo_cif.name})")
    v = py3Dmol.view(width=900, height=450, viewergrid=(1, 2))
    v.addModel(holo_cif.read_text(), "cif", viewer=(0, 0))
    _style_metal_assembly(v, panel=(0, 0))
    v.addModel(apo_cif.read_text(), "cif", viewer=(0, 1))
    _style_metal_assembly(v, panel=(0, 1))
else:
    print(f"Showing structure: {holo_cif.name}")
    v = py3Dmol.view(width=700, height=480)
    v.addModel(holo_cif.read_text(), "cif")
    _style_metal_assembly(v)

v.show()

In [ ]:
#@title 9. CPU Re-Filtering (~15s) & Export Results ZIP
import zipfile
from pathlib import Path
import hydra
import omegaconf
import pandas as pd

REFILTER_RUN_NAME = "Zn_AP_6mer_scout"  #@param {type:"string"}
NEW_BUDGET = 20  #@param {type:"integer"}
NEW_ALPHA = 0.01  #@param {type:"number"}
NEW_MIN_LIGAND_DIST = 3.5  #@param {type:"number"}
EXTRA_FILTERS = ""  #@param {type:"string"}
RUN_REFILTER = False  #@param {type:"boolean"}
EXPORT_ZIP = True  #@param {type:"boolean"}

refilter_dir = work_dir / REFILTER_RUN_NAME
if not refilter_dir.exists():
    raise FileNotFoundError(f"Run directory not found: {refilter_dir}")

if RUN_REFILTER:
    cfg_file = refilter_dir / "config" / "filtering.yaml"
    if not cfg_file.exists():
        raise FileNotFoundError(f"Missing {cfg_file}; run full pipeline first.")
    cfg = omegaconf.OmegaConf.load(cfg_file)
    cfg.design_dir = str(refilter_dir / "intermediate_designs_inverse_folded")
    cfg.outdir = str(refilter_dir)
    cfg.budget = int(NEW_BUDGET)
    if "top_budget" in cfg:
        cfg.top_budget = min(int(cfg.top_budget), int(NEW_BUDGET))
    cfg.alpha = float(NEW_ALPHA)

    extra_list = []
    if NEW_MIN_LIGAND_DIST and float(NEW_MIN_LIGAND_DIST) > 0:
        extra_list.append(
            {
                "feature": "min_ligand_distance",
                "threshold": float(NEW_MIN_LIGAND_DIST),
                "lower_is_better": False,
            }
        )
    for token in EXTRA_FILTERS.split():
        if ">" in token:
            feat, val = token.split(">", 1)
            extra_list.append({"feature": feat, "threshold": float(val), "lower_is_better": False})
        elif "<" in token:
            feat, val = token.split("<", 1)
            extra_list.append({"feature": feat, "threshold": float(val), "lower_is_better": True})
        else:
            raise ValueError(f"Invalid filter token '{token}'. Expected feature>val or feature<val.")
    cfg.additional_filters = extra_list

    print("Running Filter task on CPU...")
    task = hydra.utils.instantiate(cfg)
    task.run(cfg)
    print("Re-filtering complete!")

if EXPORT_ZIP:
    export_dir = Path("/content") if Path("/content").exists() else Path(".")
    zip_path = export_dir / f"{REFILTER_RUN_NAME}_results.zip"
    ranked_dir = refilter_dir / "final_ranked_designs"
    ranked_dir.mkdir(parents=True, exist_ok=True)
    fasta_path = ranked_dir / "top_sequences.fasta"

    metrics_csv = ranked_dir / "all_designs_metrics.csv"
    if metrics_csv.exists():
        df = pd.read_csv(metrics_csv)
        sort_col = "final_rank" if "final_rank" in df.columns else ("max_rank" if "max_rank" in df.columns else None)
        if sort_col:
            df = df.sort_values(sort_col)
        seq_col = "designed_chain_sequence" if "designed_chain_sequence" in df.columns else "designed_sequence"
        if seq_col in df.columns:
            with open(fasta_path, "w") as f:
                for _, row in df.head(int(NEW_BUDGET)).iterrows():
                    f.write(f">{row['id']}\n{row[seq_col]}\n")

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fname in ("input_material_config.yaml", "material_spec.yaml", "material_layout.json"):
            fpath = refilter_dir / fname
            if fpath.exists():
                zf.write(fpath, arcname=f"{REFILTER_RUN_NAME}/{fname}")
        if ranked_dir.exists():
            for p in ranked_dir.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path(REFILTER_RUN_NAME) / p.relative_to(refilter_dir)))

    size_mb = zip_path.stat().st_size / (1024 ** 2)
    print(f"Exported archive: {zip_path} ({size_mb:.2f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))